In [1]:
!pip install pyodbc

In [15]:
YOUR_PASSWORD = "Server_123"

In [16]:
import pyodbc

print(pyodbc.drivers())

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server']


In [17]:
import pyodbc
import json
contraseña = open("my_passwd.json", "r")
YOUR_PASSWORD = json.load(contraseña) #json es sólo el string de su contraseña.
password = YOUR_PASSWORD['my_password']
#contraseña.close()

# Set up your connection string
server = 'clase3server.database.windows.net'
database = 'sql_act'
username = 'Gael'

# Define the connection string for SQL Server (Azure)
conn_str = (
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'UID={username};'
    f'PWD={password}'
)

# Establish the connection
try:
    conn = pyodbc.connect(conn_str)
    print("Connected to the Azure SQL Database successfully!")

except pyodbc.Error as e:
    print(f"Error connecting to the database: {e}")

Connected to the Azure SQL Database successfully!


In [18]:
import pandas as pd

query_1 = "SELECT name FROM sys.tables"

cursor = conn.cursor()
cursor.execute(query_1)

tables = list(cursor.fetchall())
print(tables)

[('BuildVersion',), ('ErrorLog',), ('Address',), ('CustomerAddress',), ('SalesOrderDetail',), ('SalesOrderHeader',), ('Customer',), ('ProductModel',), ('ProductDescription',), ('Product',), ('ProductModelProductDescription',), ('ProductCategory',)]


In [19]:


query_2 = "SELECT * FROM SalesLT.SalesOrderDetail"
datos = pd.read_sql(query_2, conn)
datos



C:\Users\gael1\AppData\Local\Temp\ipykernel_55552\3642043022.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datos = pd.read_sql(query_2, conn)


,SalesOrderID,SalesOrderDetailID,OrderQty,ProductID,UnitPrice,UnitPriceDiscount,LineTotal,rowguid,ModifiedDate
0,71774,110562,1,836,356.898,0.0,356.898,E3A1994C-7A68-4CE8-96A3-77FDD3BBD730,2008-06-01
1,71774,110563,1,822,356.898,0.0,356.898,5C77F557-FDB6-43BA-90B9-9A7AEC55CA32,2008-06-01
2,71776,110567,1,907,63.900,0.0,63.900,6DBFE398-D15D-425E-AA58-88178FE360E5,2008-06-01
3,71780,110616,4,905,218.454,0.0,873.816,377246C9-4483-48ED-A5B9-E56F005364E0,2008-06-01
4,71780,110617,2,983,461.694,0.0,923.388,43A54BCD-536D-4A1B-8E69-24D083507A14,2008-06-01
...,...,...,...,...,...,...,...,...,...
537,71938,113312,3,715,29.994,0.0,89.982,64FBAAAB-9400-4664-ADD7-97115B75B27B,2008-06-01
538,71938,113313,1,881,32.394,0.0,32.394,67D4E5D4-9629-49A6-A6E0-9E1C4CDEE3DE,2008-06-01
539,71938,113314,2,875,5.394,0.0,10.788,352D06E9-8456-4966-9C73-3F227B023962,2008-06-01
540,71938,113315,3,800,672.294,0.0,2016.882,78F73142-E566-453F-A2B9-88A0FCAF4552,2008-06-01


In [20]:
df=datos.copy()

In [21]:
features= df.drop(['LineTotal','rowguid','ModifiedDate'], axis=1)
features

,SalesOrderID,SalesOrderDetailID,OrderQty,ProductID,UnitPrice,UnitPriceDiscount
0,71774,110562,1,836,356.898,0.0
1,71774,110563,1,822,356.898,0.0
2,71776,110567,1,907,63.900,0.0
3,71780,110616,4,905,218.454,0.0
4,71780,110617,2,983,461.694,0.0
...,...,...,...,...,...,...
537,71938,113312,3,715,29.994,0.0
538,71938,113313,1,881,32.394,0.0
539,71938,113314,2,875,5.394,0.0
540,71938,113315,3,800,672.294,0.0


In [22]:
labels= df['LineTotal']

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42)

In [24]:
from sklearn.linear_model import LinearRegression
reg =   LinearRegression().fit(X_train, y_train)

pred = reg.predict(X_test)

from sklearn.metrics import accuracy_score, r2_score

r2 = r2_score(pred, y_test)
r2

0.7436813906951961

In [25]:
import pickle

file = open('predict_price.pkl', 'wb')
pickle.dump(reg, file)
file.close()

In [26]:
#GETTING MY ID:
import json

id = open('my_id.json', 'r')
mi = json.load(id)

from azureml.core import Workspace

ws = Workspace.get(name="price_prediction",
                      subscription_id = mi["my_id"],
                      resource_group = "__hw1__",
                      location = "swedencentral") #centralindia

In [39]:
from azureml.core.model import Model

registered_model = Model.register(model_path="predict_price.pkl",
                                  model_name="price_prediction",
                                  workspace=ws)

Registering model price_prediction


In [51]:
print(registered_model.name)
print(registered_model.version)

price_prediction
4


In [50]:
from azureml.core import Model

models = Model.list(ws)

for m in models:
    print(m.name, m.version)

price_prediction 4
price_prediction 3
price_prediction 2
price_prediction 1


In [29]:
from azureml.core.environment import Environment
virtual_env = Environment("env-4-pricing")

from azureml.core.conda_dependencies import CondaDependencies
virtual_env.python.conda_dependencies = CondaDependencies.create(conda_packages=[
    'pandas',
    'scikit-learn'
    ])

In [46]:
from azureml.core.model import InferenceConfig
from azureml.core.webservice import AciWebservice

inference_config = InferenceConfig(
                                environment=virtual_env,
                                entry_script="score2.py",
                                )
aci_config = AciWebservice.deploy_configuration(cpu_cores=0.5, memory_gb=1)

service = Model.deploy(workspace=ws,
                        name='pricing-service',
                       models=[registered_model],
                       inference_config=inference_config,
                       deployment_config=aci_config,
                       overwrite=True,
                       )

C:\Users\gael1\AppData\Local\Temp\ipykernel_55552\4006363250.py:10: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(workspace=ws,


In [47]:
service.wait_for_deployment()

ACI service creation operation finished, operation "Succeeded"


In [48]:
scoring_uri = service.scoring_uri

In [ ]:
conn.close()